<center>
    <h1></h1>
    <h1><b>OCES 5303</b></h1>
    <h2>Assignment 2</h2>
    <hr>
    <p>Jonas Mathisrud Sterud</p>
    <p>21335836</p>
</center>

<center>
    <h1></h1>
    <h3><i>Abstract</i></h3>
    <p>
    Abstrac here.
    <small style="margin-left: 1em">[1]</small>
    </p>
    <p>
    In this assignment, we will explore the dataset, and perform ...
    </p>
    <img src="./figures/cover.jpg" width="50%">
</center>

<h1>The Data</h1>

<p>
The dataset which we'll look at in this assignment, has been supplied by Charmaine Yung <sup><small>[1]</small></sup>, from one of the papers <sup><small>[2]</small></sup> she contributed to.
</p>

<p>
It's data collected at ... using ...,
</p>

<p>
We'll explore the dataset, and create a neural network model that'll hopefully tell us a bit more about how the abundance of bacteria is affected by the temperature, seasons, etc.
</p>

In [2]:
#! This is code from the previous assignment.

#####################################
#       You might need to restart   #
#           the kernel after        #
#           running this cell.      #
#####################################

## Detect environment

try:
    import google.colab # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

## Clone repository and download dependencies
## (assumes using Conda if not in Google Colab)

if IN_COLAB:
    None
    !git clone https://github.com/jonassterud/OCES5303_A2.git
    !cp -r OCES5303_A2/* .
    %pip install -q -r ./requirements.txt
else:
    None
    %conda install -c conda-forge -c pytorch -qq --file ./requirements.txt

Channels:
 - conda-forge
 - pytorch
 - defaults
Platform: linux-64
Solving environment: ...working... done

## Package Plan ##

  environment location: /usr/local/conda/envs/OCES5303

  added / updated specs:
    - cpuonly
    - matplotlib
    - numpy
    - pandas
    - plotly
    - pytorch
    - scikit-learn
    - skorch
    - torchaudio
    - torchvision


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    skorch-1.3.1               |     pyhc455866_0         192 KB  conda-forge
    tabulate-0.10.0            |     pyhcf101f3_0          43 KB  conda-forge
    ------------------------------------------------------------
                                           Total:         235 KB

The following NEW packages will be INSTALLED:

  skorch             conda-forge/noarch::skorch-1.3.1-pyhc455866_0 
  tabulate           conda-forge/noarch::tabulate-0.10.0-pyhcf101f3_0 
  tqdm               c

In [196]:
## Imports

import numpy as np
import pandas as pd
import plotly.io as pio
import plotly.express as px
import matplotlib.pyplot as plt
import torch
import os

from plotly.subplots import make_subplots
from sklearn import set_config
from sklearn.preprocessing import MinMaxScaler, StandardScaler, FunctionTransformer, OneHotEncoder, PolynomialFeatures, SplineTransformer
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, KFold
from sklearn.dummy import DummyRegressor
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score, PredictionErrorDisplay
from sklearn.cluster import KMeans
from torch import nn
from skorch import NeuralNetRegressor

## Configuration

# set_config(transform_output = "pandas")
pio.templates.default = "plotly_dark"
seed = 9000 + 1
_ = torch.manual_seed(seed)

<h1>Parsing</h1>

<p>
The dataset, which contains 151 samples, has the following variables:
</p>

<ul>
    <li><code>YearDay</code>: Day of the year</li>
    <li><code>Bacteria_abundance</code>: Number of bacteria cells</li>
    <li><code>Projected_Daily_Insolation</code>: Solar heating</li>
    <li><code>Temp</code>: Sea Temperature (℃)</li>
    <li><code>MLLW</code>: Low of the tide TODO?</li>
    <li><code>Salinity</code>: Practical Salinity (PSU)</li>
    <li><code>Oxygen_Saturation</code>: ...</li>
    <li><code>pH</code>: pH level.</li>
    <li><code>DIC</code>: ...</li>
    <li><code>Chlorophyll</code>: ...</li>
    <li><code>NH4</code>: ...</li>
    <li><code>NO2.NO3</code>: ...</li>
    <li><code>PO4</code>: ...</li>
    <li><code>SiO4</code>: ...</li>
    
</ul>

<p>
We load the data from a <code>.txt</code> file into a Pandas <code>DataFrame</code>.
Here, we'll also drop the <code>SampleID</code> variable, since this is just an identifier for the sample, and won't be any useful for our model.
</p>

<p>
Additionaly, we see that the values of <code>YearDay</code> is linear. In reality, years are, of course, cyclical. We'll handle this later.
</p>

In [25]:
df = pd.read_csv("./data/PIDweekly_env_data.txt", sep=r"\s+")
df["YearDay"] = df["YearDay"].astype(float) 
df = df.drop(columns=["SampleID"])

<h1>Split Data</h1>

<p>
First, we'll split the data up into three disjoint subsets: train, validation and test.
</p>

<p>
We'll use the training set for explorative data analysis and training our neural network models.
</p>

<p>
To avoid overfitting (i.e. not being able to generalize), we'll keep the validation set "hidden" from the models, and only use it to check the performance of model selection, hyperparameters, etc.
</p>

<p>
Finally, to get a totally unbiased performance metric of our final model, we'll check the generalization error on our test set.
</p>

In [192]:
## Split data (70%, 15%, 15%)

X = df.drop(columns = ["Bacteria_abundance"])
y = df["Bacteria_abundance"]

X_train, X_val_test, y_train, y_val_test = train_test_split(X, y, train_size = 0.7, random_state = seed)
X_val, X_test, y_val, y_test = train_test_split(X_val_test, y_val_test, train_size = 0.5, test_size = 0.5, random_state = seed)

## Combined train set (for visualization)

Xy_train = pd.concat([X_train, y_train], axis=1)

# Use periodic spline features:
# https://scikit-learn.org/stable/auto_examples/applications/plot_cyclical_feature_engineering.html#periodic-spline-features

<h1>Exploratory Data Analysis</h1>

<p>
...
</p>

<h1>Cyclic Features</h1>

<p>
As previously mentioned, we're dealing with some cyclic data.
We have the <code>YearDay</code> variable, which tells us which day of the year the observation happened. The values here are linear, so in order to make them cyclic, we have to transform them.
</p>

<p>
Now, there's a variety of transformation methods for this, but here we'll use a periodic spline transformer. <sup><small>[3]</small></sup>
</p>

In [4]:
## Create spline transformer

_t_cyclic = SplineTransformer(
    degree=365,
    n_knots=365 + 1, # Include bias term (+ 1)
    knots=np.linspace(0, 365, 365 + 1).reshape(365 + 1, 1),
    extrapolation="periodic",
    include_bias=True,
)

t_cyclic = ColumnTransformer(
    [("cyclic", _t_cyclic, ["YearDay"])],
    remainder="passthrough",
    verbose_feature_names_out=False
)

<h1>Tensors</h1>

<p>
Now, let's create a transformer that can turn our <code>DataFrame</code> into something more suitable for Pytorch - tensors.
</p>

In [245]:
## Transformation

def _add_tensors(data):
    data_c = data.copy()
    data_c = torch.from_numpy(data_c.to_numpy()).type(torch.float32)

    return data_c

t_tensors = FunctionTransformer(func=_add_tensors)

# ## Transformation for y

# def _add_tensors_y(data):
#     data_c = data.copy()
#     data_c = torch.from_numpy(np.array(data_c)).type(torch.float32).view(-1, 1)

#     return data_c

# def _inverse_add_tensors_y(data):
#     data_c = data.copy()
#     data_c = pd.DataFrame(data_c)

#     return data_c

# t_tensors_y = TransformedTargetRegressor(func=_add_tensors_y, inverse_func=_inverse_add_tensors_y)

<h1>Neural Network</h1>

<p>
Now, let's create our neural network model.
</p>

In [ ]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.head = nn.Sequential(
            nn.Linear(13, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
        )

    def forward(self, x):
        logits = self.head(x)
        

        return logits

r_nn = NeuralNetRegressor(
    NeuralNetwork,
    max_epochs=50,
    lr=0.001,
    # Shuffle training data on each epoch
    iterator_train__shuffle=True,
)

<h1>Pipeline</h1>

<p>
...
</p>

In [256]:
pipeline = Pipeline([
    ('tensors', t_tensors), # Need to call explicitly for y
    ('regression', r_nn),
])

pipeline = pipeline.fit(X_train, t_tensors.transform(y_train).view(-1, 1))

  epoch           train_loss                              valid_loss     dur
-------  -------------------  --------------------------------------  ------
      1  11004710748160.0000  722925879699175035361681706319872.0000  0.0030
      2  710157379277723346525036233097216.0000  2834460581235449526281568256.0000  0.0028
      3  2834460581235449526281568256.0000  2823133690078381502883692544.0000  0.0027
      4  2823133690078381502883692544.0000  2811852546846616279173824512.0000  0.0027
      5  2811852251698711099820998656.0000  2800615970948533137740660736.0000  0.0021
      6  2800616561244343496446312448.0000  2789425142975752795995504640.0000  0.0019
      7  2789424847827847616642678784.0000  2778278587188749357174226944.0000  0.0018
      8  2778278292040844177821401088.0000  2767176598735428000629653504.0000  0.0020
      9  2767176303587522821276827648.0000  2756118882467883547008958464.0000  0.0014
     10  2756118292172073188303306752.0000  2745104848090305637606490112.000

<h1>References</h1>

<p>[1] Yung Lab. (2026). <a href="https://www.charmaineyung.com/" target="_blank">https://www.charmaineyung.com/</a> (Accessed online: 30.03.2026)</p>
<p>[2] Ward, C., Yung, CM., Davis, K. et al. Annual community patterns are driven by seasonal switching between closely related marine bacteria. ISME J 11, 1412–1422 (2017). <a href="https://doi.org/10.1038/ismej.2017.4" target="_blank">https://doi.org/10.1038/ismej.2017.4</a> (Accessed online: 30.03.2026)</p>
<p>[3] scikit-learn developers. (2026). <a href="https://scikit-learn.org/stable/auto_examples/applications/plot_cyclical_feature_engineering.html" target="_blank">https://scikit-learn.org/stable/auto_examples/applications/plot_cyclical_feature_engineering.html</a> (Accessed online: 30.03.2026)</p>